# fabric-rlm API tour

This notebook is a practical map of the stable public API. It starts with the two construction styles, then adds typed outputs, validation, files, skills, result diagnostics, and production safety controls.

| Need | Recommended API |
|---|---|
| Concise input/output contract | `RLM("input -> output")` |
| Natural-language task with named inputs | `RLM.task(...)` or `RLM.from_task(...)` |
| Files in the worker | `File(...)` and `LocalArtifactStore` |
| Domain playbooks | `skills=[...]`, `list_skills()`, `SkillLoader` |
| Enforced output shape | `outputs={"field": type}` and `output_validator=` |
| Run diagnostics | `RLMResult.outputs`, `.turns`, and `.report()` |
| Fabric-hosted models | `FabricLM(...)` |
| Worker guardrails | `SecurityPolicy`, `block_network`, timeouts, and recovery |

In [ ]:
%pip install -q "fabric-rlm==0.4.1"

In [ ]:
from pathlib import Path

import fabric_rlm
from fabric_rlm import (
    FabricLM,
    File,
    LocalArtifactStore,
    RLM,
    SkillLoader,
    chain,
    assert_in_range,
    list_skills,
)
from fabric_rlm.security import SecurityPolicy

print("fabric-rlm", fabric_rlm.__version__)
print("bundled skills", list_skills())

## 1. Choose a model backend

`FabricLM` uses the Fabric capacity's hosted model endpoint, so this notebook needs no API key. Outside Fabric, use `OpenAILM`, `AnthropicLM`, or a provider string supported by DSPy/LiteLLM.

```python
from fabric_rlm import OpenAILM, AnthropicLM
openai_lm = OpenAILM("gpt-4o-mini")       # reads OPENAI_API_KEY
anthropic_lm = AnthropicLM("claude-sonnet-4-5")  # reads ANTHROPIC_API_KEY
```

In [ ]:
lm = FabricLM("gpt-5.1", reasoning_effort="low")

## 2. Signature API: the shortest path

Use a signature when the task is obvious from field names. Calling an `RLM` instance is shorthand for `.run(...)`.

In [ ]:
signature_rlm = RLM("numbers -> total", lm=lm, max_turns=4)
signature_result = signature_rlm(numbers=[3, 5, 8, 13])
signature_result.outputs

## 3. Task API: natural language, typed outputs, and validation

`RLM.task` is the recommended constructor for real work. `RLM.from_task` is the explicit alias with the same contract. A list such as `outputs=["answer"]` checks names only; a mapping enforces exact runtime types and gives the model repair feedback. Validators add business rules beyond type checks.

In [ ]:
sales = [120, 80, 240, 60]
validator = chain(
    assert_in_range("largest", 0, 10_000),
    assert_in_range("average", 0, 10_000),
)

rlm = RLM.task(
    task="Use Python to return the largest sale and the arithmetic mean.",
    inputs={"sales": sales},
    outputs={"largest": int, "average": float},
    output_validator=validator,
    lm=lm,
    max_turns=5,
)

result = rlm.run()
result.outputs

Inputs can be bound in the constructor, supplied later with `rlm.run({"name": value})`, or passed through call syntax as `rlm(name=value)`. Later values override constructor-bound values, which makes one task reusable across datasets.

## 4. File inputs, artifact storage, and skills

`File` sends a lightweight path handle into the persistent worker instead of placing file contents in the prompt. `LocalArtifactStore` gives runs a predictable output directory.

Skills are Markdown playbooks that teach the model reliable domain workflows. `list_skills()` lists the bundled skills. To add your own, place `<skill-name>.md` in a directory under Lakehouse `Files`, construct `SkillLoader(skill_dir=...)`, and pass both the skill name and loader to `RLM`. Custom directories layer over the bundled skills, so both can be used in one run.

In [ ]:
lakehouse_files = Path("/lakehouse/default/Files")
tour_root = lakehouse_files / "fabric_rlm_api_tour" if lakehouse_files.exists() else Path.cwd() / ".fabric_rlm_api_tour"
store = LocalArtifactStore(tour_root)
store.write_text(
    "regional_sales.csv",
    "region,revenue\nNorth,300\nSouth,450\nWest,225\n",
)
csv_file = File(store.path("regional_sales.csv"))

# In Fabric this resolves under /lakehouse/default/Files. The local fallback
# keeps the tour executable outside Fabric without changing the API.
skills_dir = tour_root / "skills"
skills_dir.mkdir(parents=True, exist_ok=True)
(skills_dir / "regional_sales_rules.md").write_text(
    """---
applies_when:
  keywords: [regional sales, revenue]
  output_fields: [top_region, total_revenue]
excludes: []
depends_on: []
specificity: domain
---
# Regional Sales Rules
Summary: Apply the reporting rules for the regional sales CSV.

## Purpose
Use this skill when summarizing the regional sales CSV.

## Contract: output fields
- `top_region` (`str`): region with the greatest revenue.
- `total_revenue` (`float`): sum of every revenue value.

## Required verifier
```python
def verify(payload):
    assert isinstance(payload.get("top_region"), str)
    assert payload["top_region"].strip()
    assert isinstance(payload.get("total_revenue"), (int, float))
    assert payload["total_revenue"] >= 0
```

## Tripwires
- Do not average the revenue column.
- Do not return the row with the smallest revenue.
- Parse revenue as numeric before aggregation.

## Invariants
- `top_region` is non-empty.
- `total_revenue` is non-negative.

## Procedure
Read the CSV, coerce revenue to numeric, compute both outputs, call `verify`, then submit.
""",
    encoding="utf-8",
)

skill_loader = SkillLoader(skill_dir=skills_dir)
custom_skill = skill_loader.load("regional_sales_rules")
print("available skills", skill_loader.list_skills())
print("loaded custom skill", custom_skill.title)

file_result = RLM.task(
    task="Inspect the CSV and return the top region and total revenue.",
    inputs={"sales_file": csv_file},
    outputs={"top_region": str, "total_revenue": float},
    skills=["data_exploration", "regional_sales_rules"],
    skill_loader=skill_loader,
    lm=lm,
    max_turns=6,
).run()
file_result.outputs

## 5. Inspect the result, not only the payload

`RLMResult` records whether the model submitted, failures, token and timing totals, every executed turn, validation repairs, and a deterministic report suitable for logs or CI.

In [ ]:
print(result.report())
result.report(as_dict=True)

In [ ]:
turn_summary = [
    {
        "turn": turn.turn,
        "submitted": turn.submitted,
        "error": turn.error,
        "seconds": turn.duration_s,
    }
    for turn in result.turns
]
turn_summary

## 6. Production controls

Model-generated code runs in a subprocess. The default policy screens dangerous code and removes secret-like environment variables from that worker. `block_network=True` additionally seals worker egress while the parent process can still call the LM. Timeouts are per worker execution; one automatic worker restart is enabled by default.

In [ ]:
policy = SecurityPolicy.default()
secure_rlm = RLM.task(
    task="Compute the row count without using network access.",
    inputs={"sales_file": csv_file},
    outputs={"row_count": int},
    lm=lm,
    security=policy,
    block_network=True,
    timeout=120,
    recover_worker_timeouts=1,
    max_turns=4,
)
print("Configured with default security, blocked worker egress, and timeout recovery.")

## 7. Where the rest of the API fits

- `SemanticModel` exposes a Fabric semantic model as an input when `semantic-link-sempy` is available.
- `SkillLoader` loads bundled and custom skill directories; `compose_skills` resolves dependencies.
- `predict_sync(...)` and `predict(...)` are available inside worker code for nested model calls.
- `verified_task(...)` compares independent runs when agreement matters more than minimum cost.
- `ReplayLM`, `ReplayInterpreter`, and `replay_trajectory(...)` support deterministic debugging.
- Excel helpers such as `add_excel_workbook_context` and `validate_target_range_sanity` support workbook-editing tasks.

For complete options and security guidance, continue with the repository `QUICKSTART.md` and `README.md`. The focused notebooks in this directory show PDF, Spark-log, spreadsheet, and multi-source workflows.